# R02 · Hyperparameter grid search (M1 Dense+MCD, M2 LSTM+MCD)

Same grid as the submitted paper (Reviewer A asked why grid search was used):

* learning rate ∈ {5e-4, 1e-3, 2e-3}, latent dimension ∈ {16, 32, 64}, dropout p ∈ {0.05, 0.10, 0.20}
* 27 combinations × 2 architectures × 2 datasets = 108 runs, seed 0, up to 100 epochs, patience 10.
* **Selection criterion: lowest validation reconstruction loss** (as in the original notebooks). It uses
  no labels and never touches the calibration or test split.

Why grid search: the space is small (27 points), so an exhaustive search is cheap, fully reproducible, and
gives every method the same tuning budget, unlike random or Bayesian search, which add their own randomness.

In [1]:
# ── Settings ──────────────────────────────────────────────────────────────
import os
WORKERS = int(os.environ.get("REV_WORKERS", 8))   # training jobs side by side on the GPU
DEVICE  = None     # None = use the GPU if there is one
RESULTS = os.environ.get("REV_RESULTS", "results")
RUN     = True     # False = skip training, only rebuild tables/figures from results/

import os, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown
from src import report, plots
from scripts.run_jobs import run_stage, stage_tags
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
os.makedirs("tables_rev", exist_ok=True)

def export(df, name):
    """Save a table as tables_rev/REV_<name>.csv and show it."""
    path = f"tables_rev/REV_{name}.csv"; df.to_csv(path); display(df); print("saved", path)

In [2]:
if RUN:
    failed = run_stage("grid", workers=WORKERS, device=DEVICE, results=RESULTS)

stage=grid  jobs=108  todo=108  workers=8  device=cuda  results=results


[1/108    2.6 min] kindae_datasetNAB_W50_archlstm_p0.05_lr0.002_latent16_epochs100_patience10_seed0_modegrid     156s


[2/108    2.7 min] kindae_datasetNAB_W50_archlstm_p0.05_lr0.0005_latent64_epochs100_patience10_seed0_modegrid     159s


[3/108    2.9 min] kindae_datasetNAB_W50_archlstm_p0.05_lr0.001_latent32_epochs100_patience10_seed0_modegrid     175s


[4/108    3.4 min] kindae_datasetNAB_W50_archlstm_p0.05_lr0.002_latent32_epochs100_patience10_seed0_modegrid     201s


[5/108    4.1 min] kindae_datasetNAB_W50_archlstm_p0.05_lr0.001_latent64_epochs100_patience10_seed0_modegrid     241s


... (100 outputs left out)

[106/108   17.2 min] kindae_datasetSKAB_W30_archdense_p0.2_lr0.0005_latent64_epochs100_patience10_seed0_modegrid       5s


[107/108   17.2 min] kindae_datasetSKAB_W30_archdense_p0.2_lr0.001_latent64_epochs100_patience10_seed0_modegrid       3s


[108/108   17.3 min] kindae_datasetNAB_W50_archdense_p0.2_lr0.001_latent16_epochs100_patience10_seed0_modegrid      78s


done in 17.3 min, 0 failed


In [3]:
df = report.load(RESULTS)
g = df[df["mode"] == "grid"][["dataset","arch","p","latent","lr","best_val","n_epochs","time_s"]]
g = g.sort_values(["dataset","arch","best_val"])
for (ds, arch), t in g.groupby(["dataset","arch"]):
    display(Markdown(f"**{ds} · {arch}** (best first)")); display(t.head(10).reset_index(drop=True))
export(g.reset_index(drop=True), "grid_search_all")

**NAB · dense** (best first)

,dataset,arch,p,latent,lr,best_val,n_epochs,time_s
0,NAB,dense,0.05,64,0.0020,0.008118,74,60.317738
1,NAB,dense,0.05,32,0.0010,0.008237,80,61.765562
2,NAB,dense,0.05,32,0.0005,0.008360,82,68.291105
3,NAB,dense,0.05,16,0.0010,0.008382,100,84.650365
4,NAB,dense,0.05,16,0.0005,0.008472,100,84.638876
5,NAB,dense,0.05,32,0.0020,0.008487,56,46.785384
6,NAB,dense,0.05,64,0.0010,0.008703,41,31.703551
7,NAB,dense,0.05,64,0.0005,0.008704,65,50.210078
8,NAB,dense,0.05,16,0.0020,0.008792,42,33.731567
9,NAB,dense,0.10,32,0.0020,0.008961,51,41.804719


**NAB · lstm** (best first)

,dataset,arch,p,latent,lr,best_val,n_epochs,time_s
0,NAB,lstm,0.05,16,0.0010,0.008981,100,333.446245
1,NAB,lstm,0.05,32,0.0020,0.009005,61,200.536549
2,NAB,lstm,0.05,16,0.0005,0.009007,100,335.517595
3,NAB,lstm,0.05,64,0.0010,0.009055,72,241.317519
4,NAB,lstm,0.20,32,0.0020,0.009103,78,257.165623
5,NAB,lstm,0.05,16,0.0020,0.009106,46,155.859663
6,NAB,lstm,0.10,16,0.0005,0.009145,100,329.168846
7,NAB,lstm,0.10,16,0.0020,0.009280,49,161.268899
8,NAB,lstm,0.10,32,0.0020,0.009291,66,217.286188
9,NAB,lstm,0.05,32,0.0005,0.009298,98,322.711913


**SKAB · dense** (best first)

,dataset,arch,p,latent,lr,best_val,n_epochs,time_s
0,SKAB,dense,0.05,64,0.0010,0.026833,100,10.868637
1,SKAB,dense,0.05,64,0.0020,0.027057,100,11.026575
2,SKAB,dense,0.05,16,0.0010,0.028226,54,5.956601
3,SKAB,dense,0.10,64,0.0010,0.028282,100,10.936481
4,SKAB,dense,0.05,64,0.0005,0.028569,45,4.928841
5,SKAB,dense,0.05,16,0.0005,0.028636,54,5.810438
6,SKAB,dense,0.05,32,0.0005,0.028677,47,5.107830
7,SKAB,dense,0.05,32,0.0010,0.028787,24,2.695968
8,SKAB,dense,0.10,64,0.0020,0.028878,60,6.677396
9,SKAB,dense,0.10,32,0.0005,0.028934,36,4.056396


**SKAB · lstm** (best first)

,dataset,arch,p,latent,lr,best_val,n_epochs,time_s
0,SKAB,lstm,0.05,64,0.0020,0.026036,100,30.794585
1,SKAB,lstm,0.05,64,0.0005,0.027002,99,30.509151
2,SKAB,lstm,0.05,32,0.0020,0.027443,66,20.481789
3,SKAB,lstm,0.10,64,0.0005,0.027928,100,30.376625
4,SKAB,lstm,0.05,16,0.0005,0.027957,97,30.106906
5,SKAB,lstm,0.05,16,0.0020,0.028583,43,13.406914
6,SKAB,lstm,0.05,32,0.0005,0.028984,51,15.789926
7,SKAB,lstm,0.10,16,0.0020,0.028986,55,17.016436
8,SKAB,lstm,0.05,32,0.0010,0.029009,31,9.715233
9,SKAB,lstm,0.05,64,0.0010,0.029024,31,9.676067


,dataset,arch,p,latent,lr,best_val,n_epochs,time_s
0,NAB,dense,0.05,64,0.0020,0.008118,74,60.317738
1,NAB,dense,0.05,32,0.0010,0.008237,80,61.765562
2,NAB,dense,0.05,32,0.0005,0.008360,82,68.291105
3,NAB,dense,0.05,16,0.0010,0.008382,100,84.650365
4,NAB,dense,0.05,16,0.0005,0.008472,100,84.638876
...,...,...,...,...,...,...,...,...
103,SKAB,lstm,0.20,32,0.0010,0.030382,43,12.990676
104,SKAB,lstm,0.20,16,0.0005,0.030402,46,14.151937
105,SKAB,lstm,0.20,16,0.0020,0.030416,36,11.192453
106,SKAB,lstm,0.20,64,0.0020,0.030466,23,6.881611


saved tables_rev/REV_grid_search_all.csv


### Selected configurations (used by R03–R05)

In [4]:
from scripts.run_jobs import grid_winners
win = pd.DataFrame(grid_winners()).T
win.index.names = ["dataset","arch"]
export(win, "grid_winners")

p     lr  latent
dataset arch                      
SKAB    dense  0.05  0.001    64.0
        lstm   0.05  0.002    64.0
NAB     dense  0.05  0.002    64.0
        lstm   0.05  0.001    16.0

saved tables_rev/REV_grid_winners.csv


**Note for the paper:** selecting by validation loss favours the smallest dropout rate, because dropout
adds noise to the reconstruction. That explains why the original search picked p = 0.05 every time.